Proste funkcje można w miarę bezproblemowo zróżniczkować symbolicznie

In [1]:
function f(x::Number, y::Number)
    x^2 + y^2
end

f (generic function with 1 method)

In [2]:
import Pkg
Pkg.add("SymEngine")
using SymEngine

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [3]:
x, y = symbols("x y")

(x, y)

In [4]:
∂f∂x = diff(f(x, y), x);
∂f∂y = diff(f(x, y), y);
∇f = Jf = [∂f∂x ∂f∂y]

1×2 Matrix{Basic}:
 2*x  2*y

W efekcie otrzymujemy funkcje, które możemy zewaluować podstawiająć konkretne wartości za symbole

In [5]:
subs.(∇f, x => 3, y => -2)

1×2 Matrix{Basic}:
 6  -4

In [6]:
∂²f∂x² = diff(∂f∂x, x)
∂²f∂xy = diff(∂f∂x, y)
∂²f∂yx = diff(∂f∂y, x)
∂²f∂y² = diff(∂f∂y, y)
Hf = [∂²f∂x² ∂²f∂xy;
      ∂²f∂yx ∂²f∂y²]

2×2 Matrix{Basic}:
 2  0
 0  2

Natomiast w ten sposób nie zróżniczkujemy dowolnego kodu:

In [7]:
function g(x, y)
    r = 1.0
    for i=1:y
        r *= x
    end
    return r
end
dgdx = diff(g(x, y), x)

LoadError: MethodError: no method matching (::Colon)(::Int64, ::Basic)
The function `Colon()` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  (::Colon)(::T, ::Any, [91m::T[39m) where T<:Real
[0m[90m   @[39m [90mBase[39m [90m[4mrange.jl:50[24m[39m
[0m  (::Colon)(::A, ::Any, [91m::C[39m) where {A<:Real, C<:Real}
[0m[90m   @[39m [90mBase[39m [90m[4mrange.jl:10[24m[39m
[0m  (::Colon)(::T, ::Any, [91m::T[39m) where T
[0m[90m   @[39m [90mBase[39m [90m[4mrange.jl:49[24m[39m
[0m  ...


In [8]:
2.0^3

8.0

A jeśli już coś zróżniczkujemy to może się okazać, że wynikowa "formuła" jest skomplikowana:

In [9]:
function Babylonian(x; N = 10)
    t = (1+x)/2
    for i = 2:N; t=(t + x/t)/2  end
    t
end

Babylonian (generic function with 1 method)

In [10]:
diff( Babylonian(x; N=4), x ) |> expand |> display

1/16 + (-1/4)*x/(1 + 2*x + x^2) + (-1/4)*x/(1/4 + (1/2)*x + 2*x/(1 + x) + 4*x^2/(1 + x)^2 + 2*x^2/(1 + x) + (1/4)*x^2) + (-1/4)*x/(1/16 + (1/8)*x + (1/2)*x/(1 + x) + x/(1/2 + (1/2)*x + 2*x/(1 + x)) + x^2/(1 + x)^2 + (1/2)*x^2/(1 + x) + 4*x^2/(1/2 + (1/2)*x + 2*x/(1 + x))^2 + x^2/(1/2 + (1/2)*x + 2*x/(1 + x)) + 4*x^2/((1 + x)*(1/2 + (1/2)*x + 2*x/(1 + x))) + (1/16)*x^2) - x/((1 + x)*(1/4 + (1/2)*x + 2*x/(1 + x) + 4*x^2/(1 + x)^2 + 2*x^2/(1 + x) + (1/4)*x^2)) - x/((1 + x)*(1/16 + (1/8)*x + (1/2)*x/(1 + x) + x/(1/2 + (1/2)*x + 2*x/(1 + x)) + x^2/(1 + x)^2 + (1/2)*x^2/(1 + x) + 4*x^2/(1/2 + (1/2)*x + 2*x/(1 + x))^2 + x^2/(1/2 + (1/2)*x + 2*x/(1 + x)) + 4*x^2/((1 + x)*(1/2 + (1/2)*x + 2*x/(1 + x))) + (1/16)*x^2)) - 2*x/((1/2 + (1/2)*x + 2*x/(1 + x))*(1/16 + (1/8)*x + (1/2)*x/(1 + x) + x/(1/2 + (1/2)*x + 2*x/(1 + x)) + x^2/(1 + x)^2 + (1/2)*x^2/(1 + x) + 4*x^2/(1/2 + (1/2)*x + 2*x/(1 + x))^2 + x^2/(1/2 + (1/2)*x + 2*x/(1 + x)) + 4*x^2/((1 + x)*(1/2 + (1/2)*x + 2*x/(1 + x))) + (1/16)*x^2)) + 

Konkluzja: czasami można sobie pozwolić na symboliczne różniczkowanie, jednak bardzo częstu nie jesteśmy zainteresowani **wyrażeniem** matematycznym na (przykładowo) gradient, a bardziej interesuje nas **wartość** tego gradientu.

In [11]:
subs(diff( Babylonian(x; N=4), x ), x=>9)

159131117/954408050

In [12]:
.5/sqrt(9)

0.16666666666666666

In [13]:
159131117/954408050

0.16673279002623667